<a href="https://colab.research.google.com/github/Foysal061/Api-testing/blob/master/HFInputCSVGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# generate_input_template.py
# ============================================================================
# Generates the 24-row × 46-column input CSV needed by the ED Arrival LSTM.
#
# INPUTS  : VestfoldTriageReport.csv   (your raw triage data)
#           Infeksjonsdata.xlsx        (optional, monthly infection counts)
#
# APIS    : Open-Meteo  — real hourly weather  (free, no key needed)
#           Nager.Date  — Norwegian public holidays (free, no key needed)
#
# USAGE   :
#   In Colab / Jupyter:
#       from generate_input_template import generate
#       generate("VestfoldTriageReport.csv")
#
#   From terminal:
#       python generate_input_template.py --csv VestfoldTriageReport.csv
#
#   Custom window:
#       generate("VestfoldTriageReport.csv", end_datetime="2024-09-15 18")
#
# INSTALL :
#   pip install pandas numpy openpyxl requests openmeteo-requests requests-cache retry-requests
# ============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

# ── Hospital coordinates (Vestfold) ──────────────────────────────────────────
LAT      = 59.2725
LON      = 10.4184
TIMEZONE = "Europe/Oslo"
LOOKBACK = 24          # rows the model expects  — change if you retrained with 48

# ── Column order must match exactly what the model was trained on ─────────────
FEATURE_COLS = [
    "duration_mean", "duration_std", "first_doctor_response", "first_triage",
    "temperature", "humidity", "precipitation", "pressure", "wind_speed",
    "infection_rate_hourly", "is_holiday",
    "hour", "hour_sin", "hour_cos",
    "day", "dayofweek", "week", "month", "year", "day_of_year",
    "is_weekend", "is_monday", "is_friday",
    "day_sin", "day_cos", "dayofweek_sin", "dayofweek_cos",
    "week_sin", "week_cos", "month_sin", "month_cos",
    "time_of_day", "shift",
    "same_hour_last_week", "same_hour_2weeks_ago", "diff_from_last_week",
    "arrival_change_1h", "arrival_change_3h",
    "arrival_pct_change_1h", "arrival_pct_change_24h",
    "ema_12h", "ema_24h",
    "arrival_count_lag_1", "arrival_count_lag_6",
    "arrival_count_lag_12", "arrival_count_lag_24",
]

# ── Monthly climatological fallback (used only when Open-Meteo is unreachable)
WEATHER_FALLBACK = {            # month → (temp°C, humidity%, precip mm, pressure hPa, wind m/s)
    1:(-1.5,84,3.9,1006,4.9),  2:(-0.8,82,3.1,1010,4.7),
    3:(2.3,78,2.8,1011,4.5),   4:(7.1,72,2.2,1012,4.8),
    5:(12.4,68,1.8,1013,4.2),  6:(16.2,65,2.1,1013,3.8),
    7:(18.7,67,2.4,1012,3.5),  8:(17.9,69,2.9,1011,3.7),
    9:(13.1,75,3.4,1009,4.4),  10:(8.5,80,3.2,1008,5.1),
    11:(3.1,83,4.1,1005,5.8),  12:(0.2,85,4.8,1003,5.3),
}


# ============================================================================
# 1. LOAD & AGGREGATE TRIAGE CSV
# ============================================================================

def _load_triage(csv_path: str) -> pd.DataFrame:
    """
    Read raw triage CSV, aggregate to hourly counts + stats, and
    fill in any zero-arrival hours so lag features don't produce NaN.
    """
    print("  [1/5] Loading triage CSV …")
    df = pd.read_csv(csv_path, sep=";")
    df.columns = ["arrival", "departure", "first_doctor_response", "first_triage"]

    # Parse datetimes
    df["arrival"]   = pd.to_datetime(df["arrival"],   format="%d.%m.%Y %H:%M", errors="coerce")
    df["departure"] = pd.to_datetime(df["departure"], format="%d.%m.%Y %H:%M", errors="coerce")
    df = df.dropna(subset=["arrival", "departure"]).sort_values("arrival")

    # Oslo → UTC → timezone-naive
    df["arrival"]   = df["arrival"].dt.tz_localize(TIMEZONE).dt.tz_convert("UTC").dt.tz_localize(None)
    df["departure"] = df["departure"].dt.tz_localize(TIMEZONE).dt.tz_convert("UTC").dt.tz_localize(None)
    df["duration_hour"] = (df["departure"] - df["arrival"]).dt.total_seconds() / 3600

    # Encode categoricals (label encoding)
    for col in ["first_doctor_response", "first_triage"]:
        df[col] = df[col].fillna("Unknown").astype("category").cat.codes

    df["arrival_hour"] = df["arrival"].dt.floor("h")

    # Aggregate to hourly
    hourly = df.groupby("arrival_hour").agg(
        arrival_count         = ("arrival",               "count"),
        duration_mean         = ("duration_hour",         "mean"),
        duration_std          = ("duration_hour",         "std"),
        first_doctor_response = ("first_doctor_response", lambda x: int(x.mode().iloc[0])),
        first_triage          = ("first_triage",          lambda x: int(x.mode().iloc[0])),
    ).reset_index()
    hourly[["duration_mean", "duration_std"]] = hourly[["duration_mean", "duration_std"]].fillna(0)

    # Reindex to complete hourly range — fills zero-arrival gaps
    full_range = pd.date_range(hourly["arrival_hour"].min(),
                               hourly["arrival_hour"].max(), freq="h")
    hourly = (hourly.set_index("arrival_hour")
                    .reindex(full_range)
                    .rename_axis("arrival_hour")
                    .reset_index())
    hourly["arrival_count"]         = hourly["arrival_count"].fillna(0).astype(int)
    hourly["duration_mean"]         = hourly["duration_mean"].fillna(0)
    hourly["duration_std"]          = hourly["duration_std"].fillna(0)
    hourly["first_doctor_response"] = hourly["first_doctor_response"].fillna(0).astype(int)
    hourly["first_triage"]          = hourly["first_triage"].fillna(0).astype(int)

    print(f"         {len(hourly)} hourly records  "
          f"({hourly['arrival_hour'].min()}  →  {hourly['arrival_hour'].max()})")
    return hourly.sort_values("arrival_hour").reset_index(drop=True)


# ============================================================================
# 2. FETCH WEATHER  (Open-Meteo archive API — free, no key)
# ============================================================================

def _fetch_weather(start_dt: pd.Timestamp, end_dt: pd.Timestamp) -> pd.DataFrame | None:
    """
    Returns a DataFrame with columns:
        arrival_hour, temperature, humidity, precipitation, pressure, wind_speed
    Returns None if the API is unreachable (caller uses the fallback).
    """
    print("  [2/5] Fetching weather from Open-Meteo …")
    try:
        import openmeteo_requests
        import requests_cache
        from retry_requests import retry

        session  = retry(requests_cache.CachedSession(".omcache", expire_after=-1),
                         retries=5, backoff_factor=0.3)
        client   = openmeteo_requests.Client(session=session)
        response = client.weather_api(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude":    LAT,
                "longitude":   LON,
                "start_date":  start_dt.strftime("%Y-%m-%d"),
                "end_date":    end_dt.strftime("%Y-%m-%d"),
                "timezone":    "UTC",
                "hourly":      ["temperature_2m", "relative_humidity_2m",
                                 "precipitation", "surface_pressure", "wind_speed_10m"],
            },
        )[0].Hourly()

        timestamps = pd.date_range(
            start    = pd.to_datetime(response.Time(),    unit="s"),
            end      = pd.to_datetime(response.TimeEnd(), unit="s"),
            freq     = pd.Timedelta(seconds=response.Interval()),
            inclusive= "left",
        )
        df = pd.DataFrame({
            "arrival_hour": pd.to_datetime(timestamps).tz_localize(None),
            "temperature":  np.round(response.Variables(0).ValuesAsNumpy().astype(float), 2),
            "humidity":     np.round(response.Variables(1).ValuesAsNumpy().astype(float), 2),
            "precipitation":np.round(response.Variables(2).ValuesAsNumpy().astype(float), 2),
            "pressure":     np.round(response.Variables(3).ValuesAsNumpy().astype(float), 2),
            "wind_speed":   np.round(response.Variables(4).ValuesAsNumpy().astype(float), 2),
        })
        print(f"         {len(df)} hourly weather records fetched  ✓")
        return df

    except Exception as err:
        print(f"         ⚠  Open-Meteo unreachable ({err})")
        print("         ↳  Using monthly climatological averages instead.")
        return None


def _apply_weather(hourly: pd.DataFrame, weather: pd.DataFrame | None) -> pd.DataFrame:
    """Merge real weather, or fill from monthly fallback."""
    if weather is not None:
        hourly = pd.merge_asof(
            hourly.sort_values("arrival_hour"),
            weather.sort_values("arrival_hour"),
            on="arrival_hour", direction="nearest",
        )
    else:
        for col_idx, col in enumerate(["temperature","humidity","precipitation","pressure","wind_speed"]):
            hourly[col] = hourly["arrival_hour"].dt.month.map(
                lambda m, i=col_idx: WEATHER_FALLBACK[m][i]
            )
    return hourly


# ============================================================================
# 3. FETCH HOLIDAYS  (Nager.Date API — free, no key)
# ============================================================================

def _fetch_holidays(years: list[int]) -> set:
    """Return set of Norwegian public holiday dates."""
    print("  [3/5] Fetching Norwegian public holidays …")
    dates = set()
    for year in years:
        try:
            resp = requests.get(
                f"https://date.nager.at/api/v3/PublicHolidays/{year}/NO",
                timeout=10,
            )
            if resp.status_code == 200:
                for h in resp.json():
                    dates.add(pd.to_datetime(h["date"]).date())
        except Exception:
            pass

    if dates:
        print(f"         {len(dates)} holiday dates fetched for {years}  ✓")
    else:
        # Hard-coded fallback covering the dataset range
        print("         ⚠  API unreachable — using built-in Norwegian holidays.")
        fallback = [
            "2023-01-01","2023-04-06","2023-04-07","2023-04-09","2023-04-10",
            "2023-05-01","2023-05-18","2023-05-17","2023-05-28","2023-05-29",
            "2023-12-25","2023-12-26",
            "2024-01-01","2024-03-28","2024-03-29","2024-03-31","2024-04-01",
            "2024-05-01","2024-05-09","2024-05-17","2024-05-19","2024-05-20",
            "2024-12-25","2024-12-26",
        ]
        dates = {pd.to_datetime(d).date() for d in fallback}
    return dates


# ============================================================================
# 4. LOAD INFECTION DATA  (optional Excel file)
# ============================================================================

def _load_infection(xlsx_path: str, index: pd.DatetimeIndex) -> pd.Series:
    """
    Spread monthly infection totals to hourly rates.
    Falls back to a constant 0.12 if the file is missing or unreadable.
    """
    print("  [4/5] Loading infection data …")
    if not xlsx_path or not Path(xlsx_path).exists():
        print(f"         ⚠  '{xlsx_path}' not found — using 0.12 per hour.")
        return pd.Series(0.12, index=index, name="infection_rate_hourly")

    try:
        monthly = pd.read_excel(xlsx_path, header=0)
        monthly["Month"]      = pd.to_datetime(monthly["Month"], format="%b-%y")
        monthly["year_month"] = monthly["Month"].dt.to_period("M")

        rates = {}
        for ts in index:
            ym  = pd.Period(ts, "M")
            row = monthly[monthly["year_month"] == ym]
            if not row.empty:
                total  = float(row["Total_Infected_Patient_Monthly"].iloc[0])
                days   = pd.Timestamp(ts).days_in_month
                rates[ts] = total / (days * 24)
            else:
                rates[ts] = 0.12

        series = pd.Series(rates, name="infection_rate_hourly")
        print(f"         Range: {series.min():.4f} – {series.max():.4f} per hour  ✓")
        return series

    except Exception as err:
        print(f"         ⚠  Could not read Excel ({err}) — using 0.12 per hour.")
        return pd.Series(0.12, index=index, name="infection_rate_hourly")


# ============================================================================
# 5. FEATURE ENGINEERING  (mirrors training pipeline exactly)
# ============================================================================

def _engineer(
    hourly:    pd.DataFrame,
    weather:   pd.DataFrame | None,
    holidays:  set,
    infection: pd.Series,
) -> pd.DataFrame:
    """Add all 46 model features to the hourly DataFrame."""
    print("  [5/5] Engineering features …")
    df = _apply_weather(hourly, weather)

    # Infection rate
    df = df.set_index("arrival_hour")
    df["infection_rate_hourly"] = infection.reindex(df.index, fill_value=0.12)
    df = df.reset_index()

    # Holiday flag
    df["is_holiday"] = df["arrival_hour"].dt.date.isin(holidays).astype(int)

    # ── Temporal ──────────────────────────────────────────────────────────────
    df["hour"]        = df["arrival_hour"].dt.hour
    df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24)
    df["day"]         = df["arrival_hour"].dt.day
    df["dayofweek"]   = df["arrival_hour"].dt.dayofweek
    df["week"]        = df["arrival_hour"].dt.isocalendar().week.astype(int)
    df["month"]       = df["arrival_hour"].dt.month
    df["year"]        = df["arrival_hour"].dt.year
    df["day_of_year"] = df["arrival_hour"].dt.dayofyear
    df["is_weekend"]  = df["dayofweek"].isin([5, 6]).astype(int)
    df["is_monday"]   = (df["dayofweek"] == 0).astype(int)
    df["is_friday"]   = (df["dayofweek"] == 4).astype(int)

    for col, period in [("day",31), ("dayofweek",7), ("week",52), ("month",12)]:
        df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

    df["time_of_day"] = pd.cut(df["hour"], bins=[-1,6,12,18,24],
                                labels=[0,1,2,3]).astype(int)
    df["shift"]       = pd.cut(df["hour"], bins=[-1,8,16,24],
                                labels=[0,1,2]).astype(int)

    # ── Lag / rolling ─────────────────────────────────────────────────────────
    ac = df["arrival_count"]
    df["same_hour_last_week"]    = ac.shift(168)
    df["same_hour_2weeks_ago"]   = ac.shift(336)
    df["diff_from_last_week"]    = ac - df["same_hour_last_week"]
    df["arrival_change_1h"]      = ac.diff(1)
    df["arrival_change_3h"]      = ac.diff(3)

    # pct_change: replace inf (÷0) and NaN (0÷0) with 0
    df["arrival_pct_change_1h"]  = (ac.pct_change(1)
                                      .replace([np.inf, -np.inf], 0)
                                      .fillna(0))
    df["arrival_pct_change_24h"] = (ac.pct_change(24)
                                      .replace([np.inf, -np.inf], 0)
                                      .fillna(0))
    df["ema_12h"] = ac.ewm(span=12).mean()
    df["ema_24h"] = ac.ewm(span=24).mean()

    for lag in [1, 6, 12, 24]:
        df[f"arrival_count_lag_{lag}"] = ac.shift(lag)

    print(f"         Done — {len(df.columns)} columns total")
    return df


# ============================================================================
# PUBLIC API
# ============================================================================

def generate(
    csv_path:       str  = "/content/drive/MyDrive/Colab Notebooks/VestfoldTriageReport.csv",
    infection_path: str  = "/content/drive/MyDrive/Colab Notebooks/Infeksjonsdata.xlsx",
    output_path:    str  = "input_template_filled.csv",
    end_datetime:   str  = "2024-10-25",
    lookback:       int  = LOOKBACK,
) -> pd.DataFrame:
    """
    Full pipeline: CSV + APIs → filled input_template CSV.

    Parameters
    ----------
    csv_path        : path to VestfoldTriageReport.csv
    infection_path  : path to Infeksjonsdata.xlsx  (pass "" to skip)
    output_path     : where to write the output CSV
    end_datetime    : last hour of the window, e.g. "2024-10-01 12"
                      (default: "2024-10-25" — last day of the dataset)
    lookback        : number of rows (default 24)

    Returns
    -------
    pd.DataFrame    : the filled 24 × 46 feature window (also saved to output_path)
    """
    print("\n" + "=" * 62)
    print("  ED ARRIVAL LSTM  —  Input Template Generator")
    print("=" * 62)

    # 1. Load triage
    hourly = _load_triage(csv_path)

    # Determine window end — parse date-only strings to end-of-day
    end_dt = pd.to_datetime(end_datetime).floor("h")
    if end_dt.hour == 0 and ":" not in str(end_datetime).split(" ")[-1]:
        # date-only string like "2024-10-25" → use last available hour that day
        day_end = end_dt + pd.Timedelta(hours=23)
        end_dt  = min(day_end, hourly["arrival_hour"].max())

    if end_dt > hourly["arrival_hour"].max():
        raise ValueError(
            f"end_datetime {end_dt} is beyond the triage file's last record "
            f"({hourly['arrival_hour'].max()})."
        )

    window_start = end_dt - pd.Timedelta(hours=lookback - 1)
    fetch_start  = window_start - pd.Timedelta(hours=336)

    print(f"\n  Window : {window_start}  →  {end_dt}  ({lookback} rows)")
    print(f"  History: back to {fetch_start} needed for lag_24 + same_hour_2weeks_ago\n")

    # 2. Weather
    weather = _fetch_weather(fetch_start, end_dt)

    # 3. Holidays
    years    = list(range(fetch_start.year, end_dt.year + 1))
    holidays = _fetch_holidays(years)

    # 4. Infection
    infection = _load_infection(infection_path,
                                pd.DatetimeIndex(hourly["arrival_hour"]))

    # 5. Feature engineering
    full = _engineer(hourly, weather, holidays, infection)

    # Slice window
    mask   = ((full["arrival_hour"] >= window_start) &
               (full["arrival_hour"] <= end_dt))
    window = full[mask].dropna(subset=FEATURE_COLS).tail(lookback).reset_index(drop=True)

    if len(window) < lookback:
        raise ValueError(
            f"Only {len(window)} clean rows in window — need {lookback}. "
            f"Try an earlier end_datetime or ensure no large data gaps near that period."
        )

    # Final output
    result = window[FEATURE_COLS].round(6)
    result.to_csv(output_path, index=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'=' * 62}")
    print(f"  ✅  Saved → {output_path}")
    print(f"  Shape   : {result.shape[0]} rows × {result.shape[1]} columns")
    print(f"  Window  : {window_start}  →  {end_dt}")
    print()
    print(f"  Sources :")
    print(f"    Triage data   {csv_path}")
    print(f"    Weather       {'Open-Meteo API (real hourly)' if weather is not None else 'Monthly climatological means (fallback)'}")
    print(f"    Holidays      {'Nager.Date API' if holidays else 'Built-in fallback list'}")
    infection_src = infection_path if Path(infection_path).exists() else "Default 0.12/hr"
    print(f"    Infection     {infection_src}")
    print()
    print("  Preview (first 3 rows, key columns):")
    preview_cols = ["hour","temperature","humidity","arrival_count_lag_1",
                    "ema_24h","same_hour_last_week","is_holiday"]
    avail = [c for c in preview_cols if c in result.columns]
    print(result[avail].head(3).to_string(index=True))
    print("=" * 62 + "\n")

    return result


# ============================================================================
# ENTRY POINT — Colab-safe (no argparse, which conflicts with Jupyter)
# Edit the values below before running the cell.
# ============================================================================

if __name__ == "__main__":
    generate(
        csv_path       = "/content/drive/MyDrive/Colab Notebooks/VestfoldTriageReport.csv",
        infection_path = "/content/drive/MyDrive/Colab Notebooks/Infeksjonsdata.xlsx",
        output_path    = "input_template_filled.csv",
        end_datetime   = "2024-10-25",
        lookback       = LOOKBACK,
    )


  ED ARRIVAL LSTM  —  Input Template Generator
  [1/5] Loading triage CSV …
         9360 hourly records  (2023-10-01 11:00:00  →  2024-10-25 10:00:00)

  Window : 2024-10-24 11:00:00  →  2024-10-25 10:00:00  (24 rows)
  History: back to 2024-10-10 11:00:00 needed for lag_24 + same_hour_2weeks_ago

  [2/5] Fetching weather from Open-Meteo …
         ⚠  Open-Meteo unreachable (No module named 'openmeteo_requests')
         ↳  Using monthly climatological averages instead.
  [3/5] Fetching Norwegian public holidays …
         12 holiday dates fetched for [2024]  ✓
  [4/5] Loading infection data …
         Range: 0.7153 – 2.1196 per hour  ✓
  [5/5] Engineering features …
         Done — 48 columns total

  ✅  Saved → input_template_filled.csv
  Shape   : 24 rows × 46 columns
  Window  : 2024-10-24 11:00:00  →  2024-10-25 10:00:00

  Sources :
    Triage data   /content/drive/MyDrive/Colab Notebooks/VestfoldTriageReport.csv
    Weather       Monthly climatological means (fallback)
    Hol